## Data Loading

In [270]:
import os
import numpy as np
import pandas as pd
import re
from gensim.utils import simple_preprocess
import nltk
from nltk.corpus import stopwords, words, wordnet
from nltk.stem import PorterStemmer
from bertopic import BERTopic
from gensim.models import Word2Vec

#nltk.download('stopwords')
#nltk.download('words')
#nltk.download('averaged_perceptron_tagger_eng')
#nltk.download('maxent_ne_chunker_tab')
#nltk.download('punkt_tab')

In [254]:
seed_topic_list = [['flaunt', 'reveal', 'underwear', 'sex', 'tease', 'lure', 'entice', 'flirt', 
                    'seduce', 'affair', 'adulter', 'promiscuous', 'charm', 'sexy', 'attractive', 'glamour', 
                    'ass', 'booty', 'oversexed', 'busty', 'bra', 'panty', 'lingerie', 'teddy', 'slut', 'whore', 
                    'bimbo', 'floozy', 'bitch', 'jezebel', 'mistress', 'breast', 'cleavage', 'skank', 'cougar', 
                    'temptress'], ## hypersexualizing
                    ['calculated', 'cold', 'irrational', 'hysterical', 'inhuman', 'unnatural', 'unstable', 'scheme', 
                     'fake', 'deranged', 'tears', 'cry', 'unemotional', 'uncaring', 'liar', 'manipulative', 'malinger', 
                     'uncontrollable', 'angry', 'insincere', 'empathy', 'lie', 'bitch'], ## pathologizing
                    ['bisexual', 'queer', 'bitch', 'dyke', 'butch', 'lesbian', 'gay'], ## sexuality
                    ['nurture', 'homemaker', 'virgin', 'pure', 'sacrificial', 'cold', 'uncaring', 'unfriendly', 'unemotional',
                     'abandonment', 'neglect', 'overbearing', 'selfish', 'unnatural', 'assertive', 'ugly', 'aggressive',
                     'confrontational', 'violent', 'unfeeling', 'caring', 'friendly', 'emotional', 'maternal', 
                     'feminine', 'wife', 'lady', 'woman', 'housewife'], ## gender
                    ['liar', 'lie', 'manipulative', 'unbelievable', 'exaggerate', 'unreliable', 'inconsistent', 'hysterical', 
                     'corroborate', 'atypical', 'lies'], ## discrediting
                    ['alien', 'foreign', 'backward', 'citizenship', 'wetback', 'illegal', 'dirty', 'untrustworthy', 'exotic', 
                     'diverse', 'different', 'ethnic', 'accent', 'english', 'custom', 'culture', 'steal', 'infiltrate', 'border', 'home'], ## immigration
                    ['poor', 'welfare', 'poverty', 'lazy', 'cheap', 'dirty', 'disorganized', 'messy', 'greedy', 'materialistic', 
                     'acquisitive', 'opportunity', 'trailer', 'trash'], ## poverty
                    ['drugs', 'cocaine', 'speed', 'marijuana', 'heroin', 'addict'], ## addiction
                    ['animal', 'savage', 'wolves', 'negro', 'feral', 'brute', 'angry', 'untamable', 'gang', 'lazy', 'welfare', 
                     'belligerent', 'angry', 'thug', 'community', 'dirty', 'ethnic', 'mammy'], ## racial tropes
                    ['child', 'abuse', 'neglect', 'caring', 'nurture', 'mom', 'mother', 'maternal', 'matriarch', 'dad', 'father', 
                     'paternal', 'patriarch', 'negligent'] ## parenting
                    ]

In [223]:
files = []
transcripts = []

path = '/Users/emmharv/Downloads/capitaltrialtranscripts/'
for file in os.listdir(path):

    if 'txt' in file:
        with open(path + file) as f:
            transcript = f.read()

            files.append(file)
            transcripts.append(transcript)

df = pd.DataFrame({'id' : files, 'transcript' : transcripts})

In [279]:
df['transcript_cleaned'] = df['transcript'].str.replace(r'[^a-zA-Z\s]', '', regex=True) ## remove characters other than alpha and whitespace
df['transcript_cleaned'] = df['transcript_cleaned'].str.lower() ## lowercase

stop = stopwords.words('english')
stop.extend(['q', 'question', 'mr', 'ms', 'court', 'judge', 'jury', 'testimony', 'witness', 'honor']) ## remove transcript artifacts, general trial words

df['transcript_cleaned'] = df['transcript_cleaned'].apply(simple_preprocess) ## tokenize
df['transcript_cleaned'] = df['transcript_cleaned'].apply(lambda x: [word for word in x if word not in (stop)]) ## stopwords

df['transcript_cleaned'] = df['transcript_cleaned'].apply(lambda x: [word for word in x if len(wordnet.synsets(word.lower())) > 0]) ## non-dictionary words

df.head()

,id,transcript,transcript_cleaned
0,Cynthia Coffman.txt,i 5148 THIS REQUIRES YOU TO KEEP TWO SEPARATE ...,"[requires, keep, two, separate, compartments, ..."
1,Sammantha Allen.txt,mind during the trial. Form your final opinion...,"[mind, trial, form, final, opinions, opportuni..."
2,Maureen McDermott.txt,".. ""YOU MAY NOT DISREGARD THE TESTIMONY OF THE...","[may, disregard, greater, number, witnesses, m..."
3,Brittany Holberg.txt,I Case 2:15-cv-00285-Z-BP Document 11-16 Filed...,"[case, document, filed, page, following, proce..."
4,Tiffany Cole.txt,"MR. MIZRAHI: Counsel. THE COURT: Yes, sir. MR....","[counsel, yes, sir, final, hours, carol, sumne..."


In [300]:
## Get top words to filter out if needed
all_words = df['transcript_cleaned'].explode().tolist()

fdist = nltk.FreqDist(all_words) 

top_k_words, _ = zip(*fdist.most_common(10000)) 
print(top_k_words)

('yes', 'right', 'okay', 'know', 'time', 'one', 'going', 'well', 'correct', 'think', 'like', 'go', 'sir', 'back', 'said', 'see', 'case', 'get', 'say', 'may', 'evidence', 'tell', 'first', 'want', 'two', 'remember', 'take', 'people', 'told', 'got', 'also', 'defendant', 'thank', 'believe', 'ask', 'way', 'us', 'went', 'point', 'come', 'make', 'things', 'house', 'mean', 'number', 'th', 'objection', 'yeah', 'day', 'call', 'theres', 'kind', 'look', 'let', 'person', 'asked', 'miss', 'exhibit', 'made', 'recall', 'sure', 'put', 'room', 'little', 'state', 'talking', 'actually', 'give', 'please', 'ever', 'talk', 'juror', 'really', 'came', 'left', 'area', 'fact', 'next', 'morning', 'good', 'need', 'record', 'saw', 'long', 'name', 'three', 'show', 'questions', 'page', 'around', 'death', 'part', 'even', 'another', 'much', 'happened', 'never', 'thing', 'cant', 'find', 'blood', 'years', 'heard', 'last', 'home', 'murder', 'information', 'says', 'still', 'work', 'different', 'called', 'crime', 'phone', '

In [ ]:
## these are mostly court/common words and names that were messing up my modeling due to appearing everywhere
common = ['yes', 'right', 'would', 'okay', 'know', 'time', 'thats', 'dont', 'one', 'going', 'well', 'correct', 'think', 'im', 'like', 
          'go', 'sir', 'back', 'said', 'see', 'case', 'get', 'didnt', 'say', 'may', 'evidence', 'could', 'tell', 'first', 'want', 'two', 
          'remember', 'take', 'people', 'told', 'got', 'also', 'defendant', 'thank', 'anything', 'believe', 'ask', 'something', 'youre', 
          'way', 'us', 'went', 'point', 'come', 'make', 'things', 'mean', 'number', 'th', 'objection', 'yeah', 'day', 'call', 
          'theres', 'kind', 'whether', 'look', 'let', 'person', 'asked', 'miss', 'exhibit', 'made', 'recall', 'sure', 'put', 
          'state', 'talking', 'actually', 'give', 'please', 'shes', 'ever', 'talk', 'juror', 'really', 'came', 'left', 'area', 
          'fact', 'next', 'morning', 'good', 'need', 'record', 'saw', 'long', 'name', 'three', 'show', 'questions', 'page', 'around', 
          'part', 'even', 'dr', 'another', 'much', 'happened', 'never', 'thing', 'cant', 'find', 'wasnt', 'years', 'heard', 'last', 
          'information', 'says', 'still', 'different', 'called', 'phone', 'statement', 'lot', 'defense', 'ill', 'read', 'saying',
          'farley', 'took', 'else', 'report', 'maybe', 'lets', 'maam', 'times', 'wanted', 'use', 'present', 'photograph', 'testified', 
          'counsel', 'understand', 'done', 'might', 'answer', 'states', 'doesnt', 'talked', 'nothing', 'found', 'able', 'many', 'hear', 
          'looking', 'detective', 'hes', 'whats', 'marked', 'trying', 'probably', 'side', 'trial', 'second', 'used', 'front', 'true', 
          'examination', 'door', 'theyre', 'knew', 'rushton', 'either', 'ive', 'youve', 'uh', 'mrs', 'particular', 'taken', 'thought', 
          'based', 'somebody', 'far', 'issue', 'seen', 'head', 'peoples', 'opinion', 'law', 'guess', 'id', 'clerk', 'proceedings', 
          'already', 'open', 'four', 'course', 'help', 'jurors', 'object', 'defendants', 'minutes', 'asking', 'given', 'direct', 
          'date', 'office', 'following', 'interview', 'indicated', 'place', 'penalty', 'scene', 'matter', 'order', 'someone', 
          'conversation', 'reason', 'today', 'later', 'oh', 'bit', 'identification', 'tape', 'started', 'hand', 'consider', 'try', 
          'witnesses', 'must', 'end', 'prior', 'leave', 'together', 'without', 'telling', 'whatever', 'ahead', 'every', 'presence', 
          'recess', 'photographs', 'wouldnt', 'getting', 'prospective', 'couple', 'isnt', 'statements', 'sustained', 'whole', 'items', 
          'away', 'hearing', 'everything', 'days', 'start', 'circumstances', 'stand', 'mind', 'always', 'mader', 'coming', 'five', 'county', 
          'type', 'courtroom', 'period', 'move', 'inside', 'document', 'basically', 'officer', 'instructions', 'item', 'reporter', 'gave', 
          'looked', 'year', 'department', 'bring', 'problem', 'testify', 'afternoon', 'pretty', 'least', 'gentlemen', 'notes', 'agree', 
          'caro', 'aware', 'since', 'received', 'exactly', 'ladies', 'verdict', 'living', 'couldnt', 'week', 'specific', 'involved', 
          'line', 'phase', 'instruction', 'records', 'stuff', 'ingber', 'words', 'anybody', 'goes', 'sort', 'contact', 'reasonable', 
          'feel', 'specifically', 'within', 'cause', 'keep', 'top', 'uhhuh', 'overruled', 'marlow', 'certainly', 'hours', 'calls', 
          'fine', 'showing', 'certain', 'means', 'haight', 'describe', 'special', 'position', 'taking', 'excuse', 'approximately', 
          'approach', 'though', 'degree', 'held', 'upon', 'sometimes', 'smith', 'close', 'met', 'november', 'tomorrow', 'terms', 
          'decision', 'picture', 'facts', 'goldstein', 'argument', 'brought', 'test', 'csr', 'form', 'mike', 'recognize', 'regarding', 
          'enough', 'regard', 'obviously', 'john', 'admitted', 'taylor', 'important', 'clear', 'months', 'thornton', 'moved', 'attorney', 
          'letter', 'fair', 'follows', 'making', 'happen', 'weve', 'filed', 'anyone', 'several', 'coffman', 'set', 'using', 'cases', 'davis', 
          'ellison', 'stay', 'discuss', 'word', 'beyond', 'best', 'andrew', 'speak', 'act', 'incident', 'situation', 'moment', 'factor', 'six', 
          'rpr', 'individual', 'comes', 'copy', 'intent', 'check', 'process', 'gonzales', 'mentioned', 'factors', 'earlier', 'wheeler', 'step', 
          'better', 'martinez', 'certified', 'belter', 'transcript', 'occurred', 'consistent', 'possible', 'sworn', 'change', 'issues', 
          'attention', 'purpose', 'along', 'explain', 'hearsay', 'california', 'indicate', 'deputy', 'request', 'idea', 'deal', 'ivan', 
          'new', 'craig', 'sitting', 'minute', 'motion', 'knowledge', 'jordan', 'lived', 'portion', 'monday', 'stop', 'april', 'michelle', 
          'previously', 'hard', 'tried', 'burkow', 'ago', 'whereupon', 'courts', 'aki', 'june', 'youll', 'relevant', 'appears', 'determine', 
          'recollection', 'attorneys', 'note', 'general', 'ten', 'backers', 'march', 'appear', 'described', 'green', 'july', 'exhibits', 'real', 
          'response', 'werent', 'vehicle', 'big', 'circumstance', 'yesterday', 'street', 'decide', 'everybody', 'investigation', 'hour', 
          'expert', 'behind', 'st', 'location', 'needed', 'takasugi', 'address', 'excused', 'havent', 'committed', 'opportunity', 'district', 
          'august', 'althaus', 'subject', 'mc', 'small', 'october', 'traci', 'early', 'seat', 'michael', 'notice', 'sentence', 'bag', 'prove',
          'ready', 'written', 'looks', 'floor', 'quite', 'sense', 'physical', 'bench', 'iii', 'property', 'conduct', 'bingham', 
          'mark', 'ones', 'patterson', 'miskovsky', 'pm', 'bottom', 'snyder', 'wait', 'evening', 'assume', 'experience', 'redirect', 
          'indicating', 'brenda', 'malone', 'pictures', 'yall', 'offer', 'robinson', 'seeing', 'shows', 
          'referring', 'officers', 'investigator', 'gets', 'james', 'amount', 'count', 'however', 'areas', 'understanding', 'wants', 
          'less', 'provided', 'cell', 'allow', 'unless', 'reports', 'members', 'yet', 'code', 'month', 'standing', 'charge', 'weeks', 
          'actual', 'clemens', 'december', 'guilt', 'cannot', 'writing', 'oclock', 'result', 'versus', 'towards', 'discussion', 'write', 
          'additional', 'luna', 'prosecution', 'scalisi', 'allowed', 'usually', 'example', 'sent', 'control', 'wrote', 'section', 'prison', 
          'pick', 'almost', 'seated', 'reflect', 'review', 'knows', 'thinking', 'sit', 'turn', 'rest', 'stated', 'spoke', 'search', 'great', 
          'effect', 'arrived', 'absolutely', 'occasion', 'paper', 'kept', 'light', 'showed', 'foundation', 'hold', 'continue', 'autopsy', 
          'numbers', 'dermott', 'event', 'telephone', 'somewhere', 'entire', 'makes', 'reading', 'friday', 'photo', 'submit', 
          'client', 'bailiff', 'bathroom', 'documents', 'cathy', 'forensic', 'sergeant', 'discussed', 'monroe', 'letters', 'video', 'piece', 
          'parker', 'answered', 'third', 'perhaps', 'rob', 'identified', 'analysis', 'giving', 'nd', 'beginning', 'walked', 'window', 'heather', 
          'etezadi', 'caused', 'veronica', 'okunwiese', 'meet', 'turned', 'conclusion', 'reference', 'possibility', 'pause', 'play', 'system', 
          'presented', 'wont', 'popkins', 'water', 'september', 'often', 'short', 'charged', 'rs', 'middleton', 'separate',  'purposes', 
          'rather', 'mccracken', 'full', 'exum', 'familiar', 'regards', 'accurate', 'shown', 'apparently', 'eight', 'joe', 'calling', 
          'kouch', 'account', 'walk', 'examined', 'testing', 'placed', 'basis', 'view', 'plan', 'located', 'become', 'sometime', 
          'happens', 'run', 'sheriffs', 'past', 'tells', 'david', 'chance', 'table', 'identify', 'visit', 'follow', 'proof', 'testifying', 
          'west', 'duly', 'behalf', 'center', 'level', 'available', 'shall', 'behavior', 'pay', 'card', 'seven', 'raul', 'lafferty', 'longer',
          'required', 'simply', 'story', 'neck', 'listen', 'concern', 'commit', 'expect', 'list', 'greg', 'january', 'others', 'brief', 
          'considered', 'relevance', 'sample', 'san', 'stayed', 'genny', 'return', 'exact', 'balleste', 'raise', 'opening', 'significant', 
          'middle', 'lyons', 'across', 'rule', 'cross', 'virginia', 'pulled', 'store', 'thursday', 'observed', 'appeared', 'impact', 'reasons', 
          'gieger', 'calhoun', 'difference', 'board', 'lunch', 'feet', 'late', 'february', 'wendi', 'north', 'difficult', 'howardregan', 
          'attempt', 'material', 'parole', 'removed', 'rd', 'hunter', 'takes', 'seems', 'according', 'individuals', 'similar', 'inmates', 
          'persons', 'markson', 'necessarily', 'forth', 'proceed', 'blue', 'texas', 'putting', 'pattern', 'related', 'receive', 'possibly', 
          'changed', 'picked', 'ciraolo', 'term', 'wade', 'prepared', 'spell', 'arrested', 'garage', 'provide', 'conversations', 'file', 
          'rossetti', 'support', 'crisp', 'directly', 'various', 'collected', 'refer', 'voice', 'parties', 'speculation', 'tuesday', 'send', 
          'needs', 'necessary', 'decided', 'generally', 'service', 'pass', 'common', 'concerning', 'legal', 'clearly', 'nine', 'caudill', 
          'leading', 'became', 'personality', 'requested', 'interviewed', 'unit', 'alone', 'group', 'theory', 'attempted', 'offered', 'large', 
          'submitted', 'everyone', 'conrad', 'ricky', 'immediately', 'party', 'enforcement', 'pull', 'tests', 'including', 'paragraph', 
          'macher', 'landon', 'city', 'running', 'finding', 'finish', 'none', 'throughout', 'ruling', 'speaking', 'entered', 'watch', 
          'background', 'although', 'except', 'samples', 'signed', 'corner', 'diagram', 'convicted', 'williams', 'lines', 'shirt', 'lee', 
          'seemed', 'finally', 'walking', 'results', 'missing', 'commission', 'sarinana', 'ability', 'played', 'seem', 'cindy', 'final', 
          'resumed', 'youd', 'lack', 'allen', 'patient', 'near', 'wednesday', 'acts', 'meaning', 'spent', 'likely', 'opened', 'photos', 
          'lisa', 'dusek', 'swanson', 'forward', 'events', 'narine', 'direction', 'pointing', 'garcia', 'reed', 'reviewed', 'policy', 
          'station', 'personally', 'knowing', 'nobody', 'apply', 'starting', 'initially', 'folks', 'capital', 'express', 'begin', 
          'soon', 'building', 'essentially', 'ii', 'tiffany', 'complete', 'percent', 'print', 'meeting', 'unusual', 'began', 'adams', 
          'detectives', 'standard', 'chris', 'depicted', 'rosie', 'gotten', 'normally', 'holding', 'comment', 'addition', 'harrelson', 
          'michaud', 'extent', 'objections', 'referred', 'indicates', 'completely', 'returned', 'original', 'warrant', 'sign', 'janeen', 
          'lab', 'brooke', 'continued', 'leaving', 'screen', 'deliberations', 'meant', 'nelson', 'included', 'activity', 'contained', 
          'hallway', 'intend', 'social', 'briefly', 'cargill', 'assuming', 'park', 'believed', 'bank', 'inmate', 'interviews', 
          'opinions', 'starts', 'treatment', 'dark', 'names', 'inconsistent', 'conference', 'arent', 'determination', 'weekend',
          'introduced', 'character', 'profile', 'examine', 'reported', 'karl', 'fairly', 'multiple', 'previous', 'respond', 'limited', 
          'discovery', 'mitigation', 'irrelevant', 'ways', 'walker', 'intended', 'apologize', 'ran', 'andriano', 'vague', 'single', 
          'raised', 'prints', 'harrison', 'official', 'kinds', 'field', 'remain', 'initial', 'eldridge', 'hadnt', 'matters', 'include', 
          'figure', 'context', 'thomas', 'agreed', 'daveggio', 'marshall', 'penal', 'occasions', 'admit', 'instance', 'um', 'serious', 
          'affect', 'noticed', 'anyway', 'closing', 'sat', 'especially', 'appreciate', 'proved', 'security', 'spend', 'mention', 'types', 
          'observe', 'manner', 'refresh', 'impression', 'sides', 'eventually', 'handle', 'happening', 'closed', 'leon', 'somehow', 'proper', 
          'pending', 'otherwise', 'services', 'shawna', 'theyve', 'doc', 'larson', 'suggest', 'richard', 'whos', 'due', 'whose', 'ame', 
          'accept', 'anywhere', 'learned', 'ray', 'occur', 'introduce', 'toward', 'size', 'pageid', 'solemnly', 'scott', 'ultimately', 
          'major', 'points', 'practice', 'inches', 'add', 'laboratory', 'positive', 'consideration', 'research', 'circumstantial', 'la', 
          'sortino', 'sounds', 'martin', 'particularly', 'shed', 'rebuttal', 'somewhat', 'lorraine', 'susan', 'cole', 'cvdcreba', 
          'inquire', 'bill', 'psychological', 'goforth', 'parts', 'curry', 'albert', 'devon', 'closer', 'assist', 'assigned', 
          'pavatt', 'morreale', 'stipulation', 'sufficient', 'latent', 'pieces', 'neal', 'steve', 'lower', 'world', 'till', 'listed', 
          'total', 'learn', 'charges', 'contents', 'twice', 'investigators', 'program', 'session', 'ended', 'noted', 'repeat', 'rich', 
          'hey', 'handwriting', 'ah', 'draw', 'covered', 'dates', 'email', 'sound', 'buy', 'copies', 'connection', 'marvin', 'jr', 
          'arizona', 'gives', 'gallagher', 'lead', 'answers', 'comments', 'swear', 'lives', 'straight', 'prosecutor', 'seconds', 'hall', 
          'talks', 'whenever', 'findings', 'magana', 'robert', 'forms', 'richards', 'microphone', 'deliberate', 'cover', 'graham', 
          'fingerprints', 'mulder', 'entry', 'juries', 'member', 'elements', 'lay', 'tapes', 'independent', 'christopher', 'instead', 
          'tuten', 'signature', 'valerie', 'evaluation', 'understood', 'named', 'anymore', 'collect', 'deblase', 'arrest', 'jonathan', 
          'cynthia', 'therefore', 'sheet', 'trip', 'instructed', 'carolina', 'keaton', 'agreement', 'oath', 'recording', 'relates', 
          'message', 'recovered', 'quote', 'places', 'admissible', 'conviction', 'waiting', 'di', 'noon', 'lawyer', 'suspect', 'glass', 
          'alternates', 'doctors', 'malachi', 'college', 'details', 'procedure', 'willing', 'chart', 'rules', 'checks', 'explained', 
          'action', 'data', 'description', 'depending', 'quickly', 'passed', 'moore', 'kim', 'depends', 'frank', 'finished', 
          'admission', 'douglas', 'detail', 'community', 'interest', 'justice', 'parking', 'admonition', 'underneath', 'heres', 
          'probation', 'contacted', 'tested', 'johnson', 'access', 'questioned', 'instruct', 'gloves', 'experts', 'conclude',
          'reach', 'watching', 'shouldnt', 'disregard', 'source', 'shower', 'motive', 'mason', 'national', 'hundred', 'surface', 
          'indication', 'preliminary', 'judgment', 'pointed', 'staying', 'sunday', 'permission', 'onto', 'brewer', 'raymond', 
          'questioning', 'imagine', 'regular', 'branch', 'possession', 'inaudible', 'denied', 'per', 'compare', 'recorded', 'curran', 
          'range', 'christmas', 'sic', 'series', 'foreperson', 'length', 'doors', 'nurse', 'shoe', 'deliberation', 'conclusions',
          'works', 'alternate', 'haas', 'strellis', 'credit', 'checked', 'jones', 'dinner', 'jackson', 'estimate', 'chair', 'houchin', 
          'darlie', 'completed', 'license', 'duties', 'substantial', 'master', 'clinical', 'pants', 'scope', 'hole', 'christie', 'helped', 
          'lawyers', 'accomplice', 'jensen', 'gas', 'mine', 'comparison', 'agent', 'grade', 'cvzbp', 'ordered', 'transcripts', 'typically', 
          'rottiers', 'main', 'associated', 'fit', 'team', 'quick', 'obtained', 'south', 'emergency', 'forde', 'routier', 'key', 'accurately',
          'upper', 'opposed', 'rushing', 'patients', 'observations', 'definition', 'realize', 'rephrase', 'strong', 'totally', 'rafael', 
          'verdicts', 'camera', 'involving', 'withdraw', 'resume', 'saturday', 'discussing', 'low', 'staff', 'focus', 'future', 'town', 
          'definitely', 'berger', 'riverside', 'bought', 'explanation', 'hasnt', 'sabatino', 'mosty', 'conducted', 'sheriff', 'shortly', 
          'post', 'require', 'clock', 'da', 'relate', 'pardon', 'text', 'protect', 'locked', 'class', 'caros', 'ha',
          'treated', 'planning', 'media', 'reid', 'distance', 'numerous', 'spoken', 'mail', 'released', 'scale', 'swab', 'lesser', 
          'proven', 'news', 'enter', 'twelve', 'setting', 'actions', 'natural', 'loss', 'rely', 'attached', 'weiher', 'changes',
          'clark', 'led', 'rights', 'hang', 'delozier', 'exception', 'interpretation', 'drop', 'establish', 'alfaro', 'cummings', 'choice', 
          'parked', 'east', 'pressure', 'oklahoma', 'eubanks', 'papers', 'laid', 'depict', 'role', 'walton', 'upstairs', 'remove', 'reaction', 
          'assessment', 'vanessa', 'vote', 'allegation', 'ccr', 'mary', 'jimmy', 'al', 'linda', 'receipt', 'concerns', 'driveway', 'mcentire', 
          'dropped', 'steven', 'fingerprint', 'beeson', 'andor', 'funeral', 'wanting', 'fill', 'correctly', 'schedule', 'currently', 'among', 
          'originally', 'andrews', 'fourth', 'books', 'videotape', 'demeanor', 'dalton', 'washington', 'version', 'reporting', 'easy', 'liked', 
          'bringing', 'drivers', 'brandon', 'gentleman', 'rodriguez', 'partner', 'los', 'yelling', 'besides', 'education', 'risk', 'application', 
          'match', 'summer', 'wright', 'examiner', 'simple', 'determined', 'study', 'comfortable', 'younger', 'manling', 'todd', 'subpoena', 
          'relating', 'reagan', 'facing', 'prejudicial', 'counts', 'becomes', 'daily', 'plus', 'carry', 'professional', 'chief', 'appearance', 
          'alvarez', 'requires', 'effort', 'kentucky', 'extreme', 'bunch', 'communication', 'sentencing', 'performed', 'listening', 'significance', 
          'carrington', 'forget', 'meacham', 'concluded', 'mobile', 'belief', 'reached', 'sealed', 'disagree', 'accident', 'riehl', 
          'determining', 'caught', 'sustain', 'lose', 'ride', 'william', 'desk', 'cps', 'complies', 'testifies', 'interesting', 'arrive', 
          'followed', 'kennedy', 'bradbury', 'fast', 'hell', 'jim', 'exist', 'extremely', 'element', 'flores', 'proceeding', 'base', 
          'gain', 'wear', 'jeremy', 'hed', 'population', 'benefit', 'improper', 'tina', 'miller', 'clarify', 'advised', 'portions', 
          'voir', 'dire', 'fell', 'tub', 'inch', 'fbi', 'shoulder', 'blanche', 'created', 'complex', 'grabbed', 'hanging', 'remind', 'interested', 
          'continuing', 'cells', 'judicial', 'blanket', 'private', 'characteristics', 'whatsoever', 'transfer', 'filled', 'neither', 'obtain', 
          'easier', 'stick', 'crabtree', 'towel', 'row', 'prepare', 'locations', 'developed', 'omar', 'worse', 'causing', 'firetag', 'proposed', 
          'volume', 'paying', 'rooms', 'joseph', 'instrument', 'superior', 'capacity', 'alabama', 'handed', 'labeled', 'pool', 'wi',
          'causes', 'larsen', 'keys', 'joey', 'fake', 'chase', 'false', 'briuana', 'steps', 'intentionally', 'cars', 'smaller', 'baker']

df['transcript_cleaned'] = df['transcript_cleaned'].apply(lambda x: [word for word in x if word not in (common)]) ## common words

In [172]:
## removing some proper nouns id'ed during topic modeling
nouns = ['temple', 'caljic', 'winkle', 'porte', 'evans', 'griffin', 'idabel', 
        'thf', 'ts', 'tt', 'fl', 'hf', 'yoc', 'yol', 'tn', 'yo', 'tht', 'dondell', 
        'kashmiri', 'mckague', 'leons', 'barnes', 'vanessen', 'cora', 'ps', 'ling', 
        'tsang', 'alice', 'shunling', 'lucy', 'kristine', 'oporto', 'rea', 'prudential', 
        'rojas', 'anita', 'veronicas', 'tillie', 'penny', 'toby', 'noah', 'darin', 'albanesi', 
        'bernet', 'talman', 'cvhsmskl', 'kenny', 'poole', 'shea', 'kevin', 'hatfield', 'grahams', 
        'rabil', 'loflin', 'sikes', 'lj', 'kupsch', 'ronald', 'donovan', 'casey', 'whiteside', 'zoda', 
        'brad', 'amster', 'mindz', 'khan', 'rhoades', 'novis', 'kania', 'dart', 'huntington', 
        'corinna', 'gilman', 'lucio', 'padilla', 'adelaido', 'melissa', 'mariah', 'txsd', 'cv', 
        'prasad', 'toby', 'baylor', 'zamora', 'tanya', 'anita', 'vietnamese', 'nguyen', 'vu', 'vo', 'nelsons', 'tam', 
        'riva', 'freeland', 'rick', 'shannon', 'andriana', 'antoinette', 'larre', 'jenkins', 'lacaze', 'chau', 
        'vu', 'teel', 'catlett', 'brantley', 'valeska', 'jewell', 'samuel', 'edgar', 'tierra', 'cody', 'parrish', 
        'gobble', 'phoenix', 'sammantha', 'cj', 'ammandea', 'judith', 'mckay', 'utah', 'davie', 'katrina', 'rademacher', 
        'spears', 'novis', 'bankowitz', 'bausch', 'beatty', 'margaret', 'wenda', 'quintin', 'whitmore', 'holzknecht', 'qaiser', 
        'sumners', 'alan', 'nixon', 'bruce', 'sumner', 'jacksonville', 'mizrahi', 'reggie', 'carol', 'lincoln', 
        'colinas', 'kerry', 'bruno', 'rollins', 'lucy', 'mt', 'faucette', 'covington', 'asheville', 'princeton', 
        'strickland', 'covingtons', 'carlette', 'kirkland', 'dwight', 'kroger', 'loflin', 'rabil', 'samson', 'qq',
        'norris', 'towery', 'holberg', 'farren', 'brittany', 'ou', 'sp', 'ab', 'blount', 'amarillo', 'gibson', 
        'orlando', 'ellis', 'umhm', 'jaffe', 'currin', 'pender', 'shulenberger', 'franchune', 'epps', 'hutchinson', 
        'milton', 'chavez', 'cherry', 'holden', 'jeanette', 'herb', 'edna', 'sharp', 'sue', 'penny', 'car', 'truck', 'guns', 
        'gillard', 'laura', 'priessler', 'kurt', 'judith', 'sammi', 'matthew', 'davie', 'cj', 'shaniece', 'raydale', 'gayla', 
        'fazio', 'edwards', 'milo', 'seyler', 'shawanna', 'miki', 'hackler', 'galliani', 'springs', 'palm', 'dondell', 'twyla', 
        'cal', 'arivaca', 'wedow', 'tel', 'copley', 'stonex', 'jason', 'gina', 'ivans', 'weinstein', 'montiel', 'weinsteins', 
        'mills', 'belinda', 'armando', 'oscar', 'salcido', 'bryant', 'narines', 'greeley', 'munoz', 'chung', 'thb', 'op', 'tbb', 
        'havb', 'corinna', 'bb', 'hbr', 'yup', 'shb', 'arb', 'christina', 'oregon', 'christinas', 'shaniece', 'jackie', 'kochsiek', 
        'temple', 'cooper', 'celeste', 'yoo', 'koppers', 'ronnie', 'hr', 'robbeloth']

df['transcript_cleaned'] = df['transcript_cleaned'].apply(lambda x: [word for word in x if word not in (nouns)]) ## proper nouns

In [173]:
bertdocs = (df['transcript_cleaned'].apply(lambda x : [x[i:i + 500] for i in range(0, len(x), 500)]))

docs = []
for i in range(len(bertdocs)):
    for j in range(len(bertdocs[i])):
        docs.append(' '.join(bertdocs[i][j]))

In [174]:
topic_model = BERTopic()
topics, probs = topic_model.fit_transform(docs)

topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,3288,-1_house_room_little_home,"[house, room, little, home, police, children, ...",[bob seats marshal merely scenario marshal lit...
1,0,212,0_house_room_money_road,"[house, room, money, road, van, night, little,...",[patel driver expiration daveggios sleep tired...
2,1,181,1_disorder_diagnosis_brain_mental,"[disorder, diagnosis, brain, mental, symptoms,...",[demonstrates histrionics histrionics dramatic...
3,2,113,2_mitigating_aggravating_death_life,"[mitigating, aggravating, death, life, appropr...",[rise seats seats confer enters seats sentence...
4,3,105,3_hospital_medical_doctor_symptoms,"[hospital, medical, doctor, symptoms, seizure,...",[moores plans marry solid spending meals frequ...
...,...,...,...,...,...
74,73,11,73_eler_motions_sichta_messore,"[eler, motions, sichta, messore, oo, plotkin, ...",[wrong openly exchange thoughts ideas stating ...
75,74,11,74_ligon_murder_crime_bracket,"[ligon, murder, crime, bracket, crimes, passio...",[convey purported timeline chronology evidenti...
76,75,11,75_georgia_loi_rodriquez_cini,"[georgia, loi, rodriquez, cini, stalking, asia...",[difficulties recollect memory rodriquez calen...
77,76,10,76_helen_conn_thorpe_rena,"[helen, conn, thorpe, rena, shawnas, carolyn, ...",[operating wave iength nea regret remorse conf...


In [178]:
for i in range(len(topic_model.get_topic_info()) - 1):
    top = topic_model.get_topic(i)
    print(i)
    print(" '" + top[0][0] + "', " + "'" + top[1][0] + "', " + "'" + top[2][0] + "', " + "'" + top[3][0] + "', " + "'" + top[4][0] + "', " + "'" +
          top[5][0] + "', " + "'" + top[6][0] + "', " + "'" + top[7][0] + "', " + "'" + top[8][0] + "', " + "'" + top[9][0] + "', ")

0
 'house', 'room', 'money', 'road', 'van', 'night', 'little', 'police', 'drive', 'motel', 
1
 'disorder', 'diagnosis', 'brain', 'mental', 'symptoms', 'malingering', 'depression', 'antisocial', 'doctor', 'bipolar', 
2
 'mitigating', 'aggravating', 'death', 'life', 'appropriate', 'punishment', 'doubt', 'unanimously', 'recommendation', 'argue', 
3
 'hospital', 'medical', 'doctor', 'symptoms', 'seizure', 'vision', 'care', 'nausea', 'nurses', 'vomiting', 
4
 'battered', 'domestic', 'violence', 'women', 'syndrome', 'woman', 'abuse', 'batterer', 'battering', 'womans', 
5
 'wounds', 'injuries', 'wound', 'injury', 'stab', 'skin', 'body', 'blunt', 'doctor', 'force', 
6
 'rattan', 'work', 'money', 'employees', 'mawby', 'manager', 'balance', 'worked', 'paid', 'transaction', 
7
 'woods', 'becker', 'police', 'juhasz', 'ingram', 'alarm', 'cubicle', 'restaurant', 'brake', 'crime', 
8
 'kids', 'children', 'school', 'house', 'home', 'mom', 'sister', 'family', 'mother', 'little', 
9
 'blood', 'spatter',

In [ ]:
seeded_topic_model = BERTopic(seed_topic_list=seed_topic_list, top_n_words=20, n_gram_range=(1, 2))
topics, probs = seeded_topic_model.fit_transform(docs)

seeded_topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,3148,-1_house_room_home_little,"[house, room, home, little, police, child, mur...",[extra mile rosa rosa rosa treat parents paren...
1,0,537,0_death_aggravating_mitigating_crime,"[death, aggravating, mitigating, crime, doubt,...",[evaluating credibility witnesss destroy impai...
2,1,404,1_school_home_kids_house,"[school, home, kids, house, mother, children, ...",[felt cool uncle honestly cool uncle young alc...
3,2,127,2_blood_stain_stains_spatter,"[blood, stain, stains, spatter, hair, wall, bl...",[sock white tube sock alley sock sock sock blo...
4,3,121,3_battered_woman_violence_domestic,"[battered, woman, violence, domestic, domestic...",[mother care children acquiesced mother lift a...
...,...,...,...,...,...
62,61,12,61_brain_child_subdural_injuries,"[brain, child, subdural, injuries, hematoma, h...",[child struck doctor belabor pathology patholo...
63,62,12,62_child_child abuse_murder_abuse,"[child, child abuse, murder, abuse, guilty, ma...",[appropriate bracketed language underlying off...
64,63,11,63_death_drugs_feelings death_friends,"[death, drugs, feelings death, friends, topics...",[passing discussions zero impartiality friend ...
65,64,11,64_swabs_stain_blood_semen,"[swabs, stain, blood, semen, kit, carpet, dna,...",[tight skin black leather belt materials swabs...


In [181]:
for i in range(len(seeded_topic_model.get_topic_info()) - 1):
    top = seeded_topic_model.get_topic(i)
    print(top[0][0], top[1][0], top[2][0], top[3][0], top[4][0], top[5][0], top[6][0], top[7][0], top[8][0], top[9][0], 
          top[10][0], top[11][0], top[12][0], top[13][0], top[14][0], top[15][0], top[16][0], top[17][0], top[18][0], top[19][0])

death aggravating mitigating crime doubt murder life appropriate guilty crimes argue criminal respect offense alleged language little punishment sorry aggravation
school home kids house mother children mom family sister little child parents daughter brother old live friends high high school father
blood stain stains spatter hair wall blood blood knife projected pillow shorts drops cut velocity tshirt sock bloodstain bloody bed blood spatter
battered woman violence domestic domestic violence women abuse battered woman syndrome battered women batterer battered womans womans womans syndrome battering relationship child abuser abused victim
wounds injuries wound injury skin body stab blunt doctor force burn death blood stab wound chest bruising knife incised brain tissue
jail pod facility correctional custody work classification housing female dorm disciplinary housed contraband little room commissary outside dayroom corrections truth
woods police becker cubicle alarm body ingram crime roo

In [280]:
df['transcript_cleaned_stemmed'] = df['transcript_cleaned'].apply(lambda x: [PorterStemmer().stem(word) for word in x]) ## stem
df.head()

,id,transcript,transcript_cleaned,transcript_cleaned_stemmed
0,Cynthia Coffman.txt,i 5148 THIS REQUIRES YOU TO KEEP TWO SEPARATE ...,"[requires, keep, two, separate, compartments, ...","[requir, keep, two, separ, compart, brain, spe..."
1,Sammantha Allen.txt,mind during the trial. Form your final opinion...,"[mind, trial, form, final, opinions, opportuni...","[mind, trial, form, final, opinion, opportun, ..."
2,Maureen McDermott.txt,".. ""YOU MAY NOT DISREGARD THE TESTIMONY OF THE...","[may, disregard, greater, number, witnesses, m...","[may, disregard, greater, number, wit, mere, c..."
3,Brittany Holberg.txt,I Case 2:15-cv-00285-Z-BP Document 11-16 Filed...,"[case, document, filed, page, following, proce...","[case, document, file, page, follow, proceed, ..."
4,Tiffany Cole.txt,"MR. MIZRAHI: Counsel. THE COURT: Yes, sir. MR....","[counsel, yes, sir, final, hours, carol, sumne...","[counsel, ye, sir, final, hour, carol, sumner,..."


In [291]:
word2vecdocs = (df['transcript_cleaned_stemmed'].apply(lambda x : [x[i:i + 20] for i in range(0, len(x), 20)]))

docsW2V = []
for i in range(len(word2vecdocs)):
    for j in range(len(word2vecdocs[i])):
        docsW2V.append(word2vecdocs[i][j])

print(len(docsW2V))

512848


In [292]:
w2v = Word2Vec(sentences=docsW2V, vector_size=300, window=5, min_count=1, workers=4)

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


In [293]:
vocab = w2v.wv.index_to_key
print(len(vocab))

19382


In [ ]:
n = 50

topics = ['hypersexualizing', 'pathologizing', 'sexuality', 'gender', 'discrediting', 'immigration', 'poverty', 'addiction', 
          'racial tropes', 'parenting']
for i in range(len(seed_topic_list)):
    print(topics[i])
    #present_words = [word for word in seed_topic_list[i] if word in vocab]
    present_words = [PorterStemmer().stem(word) for word in seed_topic_list[i] if word in vocab]

    counts = []
    for word in present_words:
        counts.append(word + ': ' + str(w2v.wv.get_vecattr(word, 'count')))
    print(counts)
    
    sim = w2v.wv.most_similar(positive=present_words, topn=n)#, restrict_vocab=13000)

    for w in range(n):
        print(sim[w])
    
    print()

hypersexualizing
['flaunt: 10', 'reveal: 547', 'underwear: 266', 'sex: 2124', 'lure: 220', 'flirt: 89', 'affair: 942', 'adult: 1593', 'charm: 76', 'glamour: 5', 'ass: 202', 'bra: 249', 'slut: 33', 'whore: 73', 'bimbo: 1', 'bitch: 266', 'mistress: 13', 'breast: 193', 'skank: 2', 'cougar: 1']
('punk', 0.7064510583877563)
('faceless', 0.6963807344436646)
('despis', 0.6948962211608887)
('lovabl', 0.6927652359008789)
('charismat', 0.6882348656654358)
('gd', 0.6879371404647827)
('grovel', 0.6878761649131775)
('nunneri', 0.6871497631072998)
('crap', 0.6864204406738281)
('bondag', 0.6836066842079163)
('homeboy', 0.6831664443016052)
('hawaiian', 0.6822991371154785)
('scum', 0.6822910308837891)
('gunni', 0.6805440187454224)
('goali', 0.6768472194671631)
('sydney', 0.6767528653144836)
('brazen', 0.675665020942688)
('wannab', 0.6714948415756226)
('unspoken', 0.6707366108894348)
('unsophist', 0.6665018200874329)
('nanni', 0.6651691198348999)
('misfit', 0.6643394827842712)
('sheep', 0.66095727682113